# Training A Flow (using python objects)

This tutorial covers what happens when you run python -m bilbyflow.scripts.train config.yaml — what each stage does, what the config keys control, and where to look when something goes wrong. By the end you should be able to set up a training run from scratch, understand the output, and know which knobs to turn.

If you haven't already, go through the QuickStart first — it covers installation and the CLI commands. A separate version of this notebook is for python script implementations.

## Outline

The training script runs the following steps in order:

1. Build/load banks — waveforms (intrinsic), sky (extrinsic), PSD (detector noise), and optionally a noise-segment bank
2. Construct datasets — an on-the-fly training set and a disjoint validation/test set from separate waveform banks
3. Fit the standardiser — z-scoring statistics for the input (x) and target (theta)
4. Build the NPE — embedding + flow, with the prior box set from the standardised prior bounds
5. Run diagnostics — whitening checks, PSD stats, sample inspection
6. Train — curriculum-driven, with optional auxiliary supervision and JEPA consistency
7. Save and evaluate — model weights, example posteriors, PP test

Each step is discussed below, but the important thing is that all of this is driven by one YAML config file. The script itself is just handling a bunch of internals — the numerics live in the package modules, and the config parameterises them. Most functions/classes should be fairly modular and can be swapped out as wanted. The tutorials after this detail more custom choices.

## Downloading Off-Segment Data


Before training with PSD conditioning (recommended for production), you need real detector noise segments. These are used for two things:

1. __(MAIN)__ The PSD bank: realistic power spectral densities that the embedding is conditioned on, so it learns to handle the frequency-dependent noise of real detectors rather than a single idealised curve.
2. __(VERY OPTIONAL)__ Real noise realisations (optional): actual detector noise injected during training via the JEPA consistency objective. With consistency_real_rows: false (the default and recommended setting), this is not needed and the noise bank is never built — saving tens of GB of memory. In this config, if the JEPA settings are used, then they simply ask that embedding representations for the same PSD have the same representation, reducing model capacity being spent explaining variations of the same noise distribution



### What the PSD bank does

During training, each waveform draw is whitened by a PSD sampled from the bank. The bank is built from Welch estimates over your noise segments, era-balanced, with a per-frequency median-ASD floor to prevent whitening blow-up at instrumental lines. The diversity of PSDs in the bank is what teaches the embedding that the same signal can look different under different noise conditions — without it, the model memorises one noise regime and generalises poorly.

### Downloading

Use the noise download script (needs internet — on behalf of cluster admin, use the login node on a cluster even if the internet can connect from the job nodes):

```bash
# always dry-run first to see the volume
python -m bilbyflow.scripts.download_noise --config config.yaml \
    --eras O1 O2 O3a O3b --n-per-era 5 --dry-run

# then download
python -m bilbyflow.scripts.download_noise --config config.yaml \
    --eras O1 O2 O3a O3b --n-per-era 5
```


The `--config` flag is important: it reads `sampling_frequency` and `noise_data_dir` from your training config, so the files can't mismatch the model. Incompatible noise files are dropped by the PSD bank builder, so if you don't provide _anything_ that's right (e.g. the right sample rate) it will default to the standard `Bilby` design sensitivity PSD.

5 segments per era is enough for a smoke test (~20 min). For production, 25+ per era gives better PSD diversity. Leave `--n-per-era` unset to tile every available science-mode window — but we recommend `--dry-run` first.


### Without PSD conditioning

If for whatever reason you _don't_ want to condition on PSDs, then you can set `psd_conditioning: false` in the config. No noise data is needed, no PSD bank is built, and the embedding receives only strain channels. 

The model will _work_, but likely won't generalise across observing eras for standard embedding networks. The embeddings theoretically _could_ figure out the PSDs, but it would be hard to get working (anecdotally). It is essentially feature engineering. TLDR make the networks' lives easier if you can, and providing the noise profile directly does exactly that.

## Detailed Config Descriptions

The config is a YAML file. Every key has a default except the ones marked required. Below is a semi-production config with annotations — copy it, change the paths, and it runs.

### Signal and Grid

```yaml
duration: 4.0                    # REQUIRED. Segment length [s].
                                 # Must hold the signal: at Mc=30 Msun from
                                 # 20 Hz the inspiral is ~0.8 s, so 4 s is
                                 # comfortably long. Shorter = all waveforms
                                 # NaN and the dataset is empty.

sampling_frequency: 2048         # REQUIRED. [Hz]. 2048 is standard for aLIGO;
                                 # 4096 for high-mass or higher-order modes.

f_min: 20                        # REQUIRED. Low-frequency cutoff [Hz].

waveform_approximant: IMRPhenomXPHM  # REQUIRED. LALSimulation model name.
                                     # IMRPhenomXPHM for precessing + HM.
                                     # IMRPhenomPv2 for a faster non-HM model.
                                     # should be a drop in for any waveform models implemented in Bilby

ref_geocent_time: 1126259462.4   # REQUIRED. GPS time the sky bank centres on.
                                 # Convention: GW150914's merger time. 
                                 # It shouldn't actually matter what time it is, the transforms should
                                    # all be invertible. But I (Liam) have not tested changing it very much.
                                    # So if you're fine being a beta tester, go ahead...
```

These define the analysis grid. Every frequency array, windowing function, and Welch PSD in the package derives from these five numbers. Changing them after training means retraining/ it ain't gonna work.

### Parameters and Priors

```yaml
inferred_parameters:             # REQUIRED. What the flow models.
  - chirp_mass                   # For the 12-parameter BBH model:
  - mass_ratio                   # masses, distance, inclination,
  - luminosity_distance          # sky (in detector-frame coords),
  - theta_jn                     # spins, and spin tilts/phases.
  - ra                           # ra/dec are DETECTOR-FRAME: the flow
  - dec                          # outputs (dt_HL, phi_det), and the
  - a_1                          # reweighter converts back to (ra, dec).
  - a_2
  - tilt_1
  - tilt_2
  - phi_12
  - phi_jl

nuisance_parameters:             # Marginalised in the IS target by
  - phase                        # synthetic_phase.py. They still VARY in
  - geocent_time                 # training data (sky bank draws psi into
  - psi                          # fp/fc, tc into merger position, phase
                                 # into the waveform), so the flow learns
                                 # to be invariant to them — then the
                                 # reweighter marginalises them exactly.
```

The split matters for the maths and you have a lot of choices post-training, especially when it comes to reweighting integration numerics. 

But practically it's just that: 
- parameters in inferred_parameters are flow outputs, 
- parameters in nuisance_parameters are marginalised in the reweighting target. 

Everything else (like component masses, redshift) is derived.

```yaml
priors:
  chirp_mass: {min: 10.0, max: 80.0}
  mass_ratio: {min: 0.125, max: 1.0}
  luminosity_distance: {min: 100.0, max: 5000.0}
  theta_jn: {min: 0.0, max: 3.14159}
  ra: {min: 0.0, max: 6.28318}
  dec: {min: -1.5708, max: 1.5708}
  a_1: {min: 0.0, max: 0.99}
  a_2: {min: 0.0, max: 0.99}
  tilt_1: {min: 0.0, max: 3.14159}
  tilt_2: {min: 0.0, max: 3.14159}
  phi_12: {min: 0.0, max: 6.28318}
  phi_jl: {min: 0.0, max: 6.28318}
  phase: {min: 0.0, max: 6.28318}
  geocent_time: {min: -0.11, max: 0.11}
  psi: {min: 0.0, max: 3.14159}
```

These define the min/max for sampling. The actual bilby prior families are constructed by `make_prior_dict`: 
- chirp mass and mass ratio get `UniformInComponents` (uniform in component masses), 
- distance gets `PowerLaw(alpha=2)`, 
- dec gets `Cosine`, 
- tilts get `Sine`, and 
- everything else is `Uniform`. 

The config just supplies the bounds.

### Coordinate Transforms

```yaml
dL_param: 'log'           # Train in ln(dL). [Note 1]

dL_train_alpha: -1         # TRAINING proposal only. [Note 2]

Mc_param: 'linear'         # Chirp mass coordinate: 'linear' (default)
                           # or 'log'. Linear is fine for the BBH range.
```

[Note 1]: The flow models `ln(dL)` rather than `dL` directly, because `sigma_{ln dL} ~ 1/rho` is roughly constant across SNR, the posterior width is homoscedastic in log space. 

This is a huge help for the flow, which otherwise has to learn a distance-dependent width.

[Note 2]: `-1 = log-uniform`, i.e. equal proposal mass per `dL` decade. The physical prior (`PowerLaw alpha=2`) is applied at reweighting via prior-swap SIR. This gives more loud-end coverage, where reweighting efficiency died during tests without this training prior choice. Using the 'physically motivated' prior meant that there was a large pile-up of events at high `dL` and low SNR, where events aren't seen, wasting capacity. 

The Jacobians from these transforms are tracked analytically in inference.sample and folded into log_q, so the importance weights are correct regardless of which coordinate the flow uses internally.

### Bank Sizes

For training, example data is made by combining intrinsic waveforms, sky position information and the noise.

The training waveform bank contains (h+, hx) pairs at intrinsic draws. 1M is production; 50-100k for a smoke test. Precomputed once and cached in the output directory.

The skybank contains extrinsic parameter draws (ra, dec, psi, tc, projected into fp, fc, dt_HL, phi_det). It is also cached.

```yaml
n_waveforms: 100000 

n_sky_bank: 100000   

n_standardisation: 20000   # Draws for fitting the standardiser z-scores.
n_val: 20000               # Fixed validation set size.
n_test: 20000              # Fixed test set size.
n_waveform_workers: 0      # Workers for waveform generation. 0 = auto
                           # (uses SLURM_CPUS_PER_TASK or all cores).
```

### Embedding Architecture

```yaml
embedding_type: conv1d_resnet     # Production default.

embedding_output_dim: 512         # Context vector width. This is what the
                                  # flow conditions on. 512 is validated for
                                  # 12 parameters; smaller (128-256) works
                                  # for fewer parameters or smoke tests.

conv1d_channels: [32, 64, 128, 128]  # Per-layer channel counts in the 1D stem.
conv1d_kernel: 7                      # Kernel size in the 1D convolutions.
conv1d_dropout: 0.1                   # Dropout in the embedding head.
conv1d_resnet_stem_out: 512           # Channels entering the 2D ResNet.
conv1d_resnet_backbone: resnet18      # torchvision backbone: resnet18/34/50.
```

he embedding takes the whitened strain + PSD context and produces a fixed-length context vector for the flow. The conv1d_resnet architecture is what we use in the paper: a 1D convolutional stem reduces the frequency resolution, then the features are folded into a near-square grid and fed through a 2D ResNet. One branch for frequency-domain channels, one for time-domain, plus a PSD MLP — all concatenated and projected to embedding_output_dim by a final head.

See the [StrainEmbedding tutorial](StrainEmbedding.html) for how to write your own. Some in-built options include
- `conv1d`        — two 1D conv stems + PSD MLP
- `conv1d_resnet` — 1D stem -> 2D ResNet (production)
- `resnet`        — fold all channels into one ResNet
- `fd_psd`        — FD+PSD only, no TD (ablation)
- `film`          — FiLM-modulated by PSD
- `mlp`           — flatten everything (baseline)

### PSD Conditioning


```yaml
psd_conditioning: true            # Append z-scored log-PSD to the embedding
                                  # input. Needs noise_data_dir.

noise_data_dir: /path/to/noise_data  # Where the era/ noise segments live.

psd_encoder_hidden: [512, 256, 128]  # MLP layers for the PSD encoder if it is separate in the architecture.
psd_encoder_out: 64                  # If there is a separate PSD encoder output dim, it should be 
                                     # concatenated to the strain features before the head.
psd_context_clip: 10.0               # Clip z-scored log-PSD at +-this value.
```

The PSD context is 0.5 * log10(psd), z-scored per frequency bin using statistics computed from the PSD bank, then clipped. It's concatenated channel-wise with the FD strain in the embedding, so the network can learn "this bin is noisy, downweight it" rather than treating every bin equally.

The PSD bank itself is built from Welch estimates over the noise segments:

```yaml
psd_bank:
  n_psds: 10000               # How many PSDs in the bank.
  eras: null                   # Which eras to use (null = all found).
  asd_floor_factor: 5.0       # Per-frequency ASD floor at median/factor.
  era_weights: {O1: 1, O2: 1, O3a: 1, O3b: 1}  # Balance across eras.
```

### Flow Architecture


```yaml
hidden_features: 512      # Conditioner MLP width. This is the most
                           # impactful capacity knob — wider = more
                           # expressive transforms, more parameters.

num_transforms: 16         # Number of coupling layers. More = more
                           # expressive, slower to sample. 64 is what was used
                           # in the paper (yes it's a lot, we did A/B testing
                           # and more equalled better in our case); 
                           # 4-16 for smoke tests.

num_bins: 24               # Spline knots (NSF/NCSF only). More bins =
                           # sharper splines, better at multimodal
                           # posteriors. Default 10 is sbi's; 24 is ours.

flow_dropout: 0.05         # Dropout in the conditioner MLPs. Injected
                           # via the activation factory (zuko's MaskedMLP
                           # has no dropout argument). Set to 0.0 in the
                           # final curriculum stage via final_stage_flow_dropout.

weight_decay: 0.05         # AdamW weight decay. The main regulariser
                           # alongside fresh on-the-fly noise every epoch.
```


The flow is a Neural Spline Flow (NSF) by default — rational-quadratic spline coupling transforms, backed by zuko. The coupling structure (passes=2) means the inverse (sampling) is one pass through the network, same cost as the forward (density evaluation). This is why sampling 100k draws takes ~1 second rather than minutes.

Other architectures are available (flow_type: maf, ncsf, nice, sospf, gf) but NSF with the production settings above is what we validated.

### Training and Optimization


```yaml
batch_size: 2048              # Effective batch size. If larger than
                              # micro_batch_size, gradient accumulation
                              # is used automatically.

micro_batch_size: 256         # Per-forward chunk. P100 (16 GB): 128-256.
                              # A100 (40/80 GB): set = batch_size.

learning_rate: 0.0005         # Initial LR (overridden per stage).
eta_min: 1.0e-5               # Cosine schedule minimum LR.
clip_max_norm: 10.0           # Gradient clipping.

noise_source: gaussian_physical  # Training noise model. Options:
                                 #   gaussian_physical — Gaussian noise with the
                                 #     same Tukey window as the signal (matches
                                 #     both real-data whitening and the bilby
                                 #     likelihood). Recommended.
                                 #   gaussian_whitened — unit-variance whitened
                                 #     noise (legacy, not matched).
                                 #   real — actual detector segments from the
                                 #     noise bank (needs noise_data_dir +
                                 #     large memory).

num_workers: 16               # DataLoader workers. Set <= cpus-per-task.
prefetch_factor: 4            # Batches prefetched per worker.
```

### Curriculum


```yaml
curriculum_stages:
  - {dL_max: 800,  epochs: 50,  lr: 5.0e-4, t_max: 50,  patience: 20}
  - {dL_max: 1100, epochs: 40,  lr: 5.0e-4, t_max: 40,  patience: 20}
  - {dL_max: 1500, epochs: 40,  lr: 1.0e-4, t_max: 40,  patience: 20}
  - {dL_max: 2000, epochs: 40,  lr: 1.0e-4, t_max: 40,  patience: 20}
  - {dL_max: 2500, epochs: 40,  lr: 5.0e-5, t_max: 40,  patience: 20}
  - {dL_max: 3500, epochs: 60,  lr: 5.0e-5, t_max: 60,  patience: 50}
  - {dL_max: 5000, epochs: 100, lr: 1.0e-5, t_max: 110, patience: 50}

curriculum_restandardise: x    # Re-fit x statistics at each stage
                               # (the dL cap changes the x distribution).

final_stage_flow_dropout: 0.0  # Drop all flow dropout in the last stage
                               # for sharper IS proposals.
```

The curriculum is a sequence of stages with increasing dL_max — the maximum luminosity distance in the training draws. This acts as an SNR floor: early stages see only loud events (small dL, high SNR) where the posterior is tight and the NLL gradient is strong. Later stages open up to quieter events.

Each stage has its own learning rate, cosine schedule (t_max), and early-stopping patience. The standardiser's x statistics are re-fitted at each stage boundary because the dL cap changes the signal amplitude distribution.

For a smoke test, two stages are enough:

```yaml
curriculum_stages:
  - {dL_max: 1500, epochs: 30, lr: 5.0e-4, t_max: 30, patience: 15}
  - {dL_max: 5000, epochs: 50, lr: 1.0e-4, t_max: 50, patience: 30}
  ```

### Auxiliary Supervision

```yaml
aux_supervision: true       # Train a small MLP head to predict
                            # noiseless-signal summaries from the
                            # embedding context. Shapes the embedding
                            # early when the NLL gradient is weak.

aux_lambda: 0.5             # Weight of the aux MSE in the total loss.
aux_anneal_frac: 0.7        # Lambda anneals to 0 by this fraction of
                            # each stage, so the aux objective doesn't
                            # fight the NLL at convergence.
aux_n_channels: 256         # How many context dims the aux head reads.
aux_head_hidden: 128        # Hidden layer width in the aux MLP.
```